# Experiments – Edouard

Each experiment is a DLC shuffle that holds out one session (train on the 7 others).
Needs `dlc/pupil-dani-2026-11-13/` with its `labeled-data/`. Steps already done are skipped, so "Run All" is safe.

In [ ]:
from pathlib import Path

import deeplabcut
import pandas as pd
from deeplabcut.utils import auxiliaryfunctions

PROJECT = Path("dlc") / "pupil-dani-2026-11-13"
MODELS_DIR = PROJECT / "dlc-models-pytorch" / "iteration-0"
EVAL_DIR = PROJECT / "evaluation-results-pytorch" / "iteration-0"
config_path = str(PROJECT / "config.yaml")

cfg = auxiliaryfunctions.read_config(config_path)
cfg["project_path"] = str(PROJECT.resolve())
auxiliaryfunctions.write_config(config_path, cfg)

merged = pd.read_hdf(next((PROJECT / auxiliaryfunctions.get_training_set_folder(cfg)).glob("CollectedData_*.h5")))
frame_sessions = [str(idx[1]) for idx in merged.index]
print(pd.Series(frame_sessions).value_counts().sort_index().to_string())  # 393 labeled frames over 8 sessions


def split_indices(test_session: str) -> tuple[list[int], list[int]]:
    """Return the (train, test) row indices of the merged labels, holding out one session."""
    test = [i for i, session in enumerate(frame_sessions) if session == test_session]
    train = [i for i, session in enumerate(frame_sessions) if session != test_session]
    if not test:
        raise ValueError(f"No labeled frames for session {test_session}.")
    return train, test


def shuffle_dir(shuffle: int) -> Path | None:
    """Return a shuffle's model folder, or None if the shuffle doesn't exist yet."""
    matches = list(MODELS_DIR.glob(f"*shuffle{shuffle}"))
    return matches[0] if matches else None


def create_shuffle(shuffle: int, test_session: str) -> None:
    """Create a shuffle holding out one session, unless the shuffle already exists."""
    if shuffle_dir(shuffle) is not None:
        print(f"Shuffle {shuffle} already exists, skipped.")
        return
    train, test = split_indices(test_session)
    print(f"Shuffle {shuffle}: test on {test_session} | train frames: {len(train)} | test frames: {len(test)}")
    deeplabcut.create_training_dataset(
        config_path, Shuffles=[shuffle],
        trainIndices=[train], testIndices=[test],
        net_type="resnet_50",
    )


def configure_shuffle(shuffle: int, train: dict, inference: dict, epochs: int) -> None:
    """Overwrite chosen training settings in a shuffle's pytorch_config.yaml."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    path = folder / "train" / "pytorch_config.yaml"
    pcfg = auxiliaryfunctions.read_plainconfig(path)
    pcfg["data"]["train"].update(train)
    pcfg["data"]["inference"].update(inference)
    pcfg["train_settings"]["epochs"] = epochs
    auxiliaryfunctions.write_plainconfig(path, pcfg)
    print(f"Shuffle {shuffle} | train: {train} | inference: {inference} | epochs: {epochs}")


def train_shuffle(shuffle: int, epochs: int) -> None:
    """Train a shuffle, unless its final snapshot already exists."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    if (folder / "train" / f"snapshot-{epochs:03d}.pt").exists():
        print(f"Shuffle {shuffle} already trained, skipped.")
        return
    deeplabcut.train_network(config_path, shuffle=shuffle)


def evaluate_shuffle(shuffle: int, epochs: int, force: bool = False) -> None:
    """Evaluate a shuffle's final snapshot, unless already done (never the 'best' one, picked with the test score).

    hist_eq is random at test time in DLC, so re-evaluating can change the numbers: use force=True on purpose only.
    """
    done = list(EVAL_DIR.glob(f"*shuffle{shuffle}/*_snapshot_{epochs:03d}-results.csv"))
    if done and not force:
        print(f"Shuffle {shuffle} already evaluated, skipped.")
        return
    deeplabcut.evaluate_network(
        config_path, Shuffles=[shuffle],
        snapshots_to_evaluate=[f"snapshot-{epochs:03d}"],
        per_keypoint_evaluation=True,
    )

354 39


ValueError: Cannot create shuffle 2 as it already exists - you must either create the dataset with `userfeedback=False` or delete the shuffle with index 2 manually (in `dlc-models`/`dlc-models-pytorch` and in the `training-datasets` folder) if you want to create a new shuffle with that index. You can otherwise create a shuffle with a new index. Existing indices are [1, 2, 99, 100].

## Shuffle 2 – histogram equalization

`hist_eq` in train and inference (DLC applies it to each image with probability 0.5), 100 epochs.
Result on 110535: 93 px mean error (baseline 238 px), 7 % detection (baseline 0 %).

In [ ]:
create_shuffle(2, test_session="110535")
configure_shuffle(2, train={"hist_eq": True}, inference={"hist_eq": True}, epochs=100)
train_shuffle(2, epochs=100)
evaluate_shuffle(2, epochs=100)

Training with configuration:
config_version: 0
model: backbone={'type': 'ResNet', 'model_name': 'resnet50_gn', 'output_stride': 16, 'freeze_bn_stats': False, 'freeze_bn_weights': False} heads={'bodypart': {'type': 'HeatmapHead', 'weight_init': 'normal', 'predictor': {'type': 'HeatmapPredictor', 'apply_sigmoid': False, 'clip_scores': True, 'location_refinement': True, 'locref_std': 7.2801}, 'target_generator': {'type': 'HeatmapGaussianGenerator', 'num_heatmaps': 8, 'pos_dist_thresh': 17, 'heatmap_mode': 'KEYPOINT', 'gradient_masking': False, 'generate_locref': True, 'locref_std': 7.2801}, 'criterion': {'heatmap': {'type': 'WeightedMSECriterion', 'weight': 1.0}, 'locref': {'type': 'WeightedHuberCriterion', 'weight': 0.05}}, 'heatmap_config': {'channels': [2048, 8], 'kernel_size': [3], 'strides': [2]}, 'locref_config': {'channels': [2048, 16], 'kernel_size': [3], 'strides': [2]}}} backbone_output_channels=2048 neck=None pose_model=None
net_type: resnet_50
method: bu
device: auto
metadata:

## Shuffle 3 – histogram equalization + horizontal flip

Same as shuffle 2, plus `hflip: 0.5` in training only (no keypoint swap: labels are anatomical).
Result on 110535: 21 px mean error, 36 % detection; the anterior/posterior swap is gone.

In [ ]:
create_shuffle(3, test_session="110535")
configure_shuffle(3, train={"hist_eq": True, "hflip": 0.5}, inference={"hist_eq": True}, epochs=100)
train_shuffle(3, epochs=100)
evaluate_shuffle(3, epochs=100)

NameError: name 'create_shuffle' is not defined

## Development rounds – 135059 and 171101 held out

Does shuffle 3's gain hold on other sessions? Each round compares DLC's defaults with `hist_eq` + flip, both at 100 epochs.

| Shuffle | Held out | Settings |
|---|---|---|
| 4 | 135059 | DLC defaults |
| 5 | 135059 | hist_eq + hflip |
| 6 | 171101 | DLC defaults |
| 7 | 171101 | hist_eq + hflip |

About 2 h 30 of training in total.

In [ ]:
BASELINE: dict[str, dict] = {"train": {}, "inference": {}}
HISTEQ_FLIP: dict[str, dict] = {"train": {"hist_eq": True, "hflip": 0.5}, "inference": {"hist_eq": True}}

DEV_RUNS: dict[int, tuple[str, dict[str, dict]]] = {
    4: ("135059", BASELINE),
    5: ("135059", HISTEQ_FLIP),
    6: ("171101", BASELINE),
    7: ("171101", HISTEQ_FLIP),
}

for shuffle, (test_session, settings) in DEV_RUNS.items():
    create_shuffle(shuffle, test_session=test_session)
    configure_shuffle(shuffle, train=settings["train"], inference=settings["inference"], epochs=100)

In [ ]:
for shuffle in DEV_RUNS:
    train_shuffle(shuffle, epochs=100)

In [ ]:
for shuffle in DEV_RUNS:
    evaluate_shuffle(shuffle, epochs=100)